# World Cup 2026 — Exploratory Data Analysis
**Notebook 01** · Historical match results (1872–2026)

Goals of this notebook:
- Understand the shape and quality of the data
- Check for missing values and anomalies
- Answer basic football questions that inform model features
- Identify World Cup matches specifically

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('dark_background')
pd.set_option('display.max_columns', None)
print('Libraries loaded ✓')

## 1. Load the Data

In [ ]:
results = pd.read_csv('../data/raw/historical/results.csv')
goalscorers = pd.read_csv('../data/raw/historical/goalscorers.csv')
shootouts = pd.read_csv('../data/raw/historical/shootouts.csv')

print(f'results.csv      → {len(results):,} rows')
print(f'goalscorers.csv  → {len(goalscorers):,} rows')
print(f'shootouts.csv    → {len(shootouts):,} rows')

## 2. Basic Sanity Checks

In [ ]:
# Shape and columns
print('--- results.csv ---')
print(results.info())
print('\nFirst 5 rows:')
results.head()

In [ ]:
# Missing values
print('Missing values per column:')
print(results.isnull().sum())

# Date range
results['date'] = pd.to_datetime(results['date'])
print(f'\nDate range: {results["date"].min().year} → {results["date"].max().year}')

## 3. Tournament Breakdown

In [ ]:
# What tournaments are in the dataset?
print('Top 20 tournaments by match count:')
print(results['tournament'].value_counts().head(20))

In [ ]:
# Isolate World Cup matches only
wc = results[results['tournament'] == 'FIFA World Cup'].copy()
print(f'Total World Cup matches in dataset: {len(wc)}')
print(f'World Cup editions covered: {wc["date"].dt.year.nunique()}')
wc.head()

## 4. Goals Analysis

In [ ]:
# Average goals per game — all matches
results['total_goals'] = results['home_score'] + results['away_score']
print(f'Average goals per game (all matches): {results["total_goals"].mean():.2f}')
print(f'Average goals per game (World Cup only): {wc["home_score"].add(wc["away_score"]).mean():.2f}')

In [ ]:
# Goals per decade — has the game changed?
results['decade'] = (results['date'].dt.year // 10) * 10
goals_by_decade = results.groupby('decade')['total_goals'].mean()

plt.figure(figsize=(12, 4))
goals_by_decade.plot(kind='bar', color='#00e5ff')
plt.title('Average Goals Per Game by Decade')
plt.xlabel('Decade')
plt.ylabel('Avg Goals')
plt.tight_layout()
plt.show()

## 5. Win / Draw / Loss Rates

In [ ]:
def get_result(row):
    if row['home_score'] > row['away_score']:
        return 'Home Win'
    elif row['home_score'] < row['away_score']:
        return 'Away Win'
    else:
        return 'Draw'

results['result'] = results.apply(get_result, axis=1)
wc['result'] = wc.apply(get_result, axis=1)

print('--- All Matches ---')
print(results['result'].value_counts(normalize=True).mul(100).round(1).astype(str) + '%')

print('\n--- World Cup Only ---')
print(wc['result'].value_counts(normalize=True).mul(100).round(1).astype(str) + '%')

## 6. Most Active Teams

In [ ]:
# Teams with most international matches
home_counts = results['home_team'].value_counts()
away_counts = results['away_team'].value_counts()
total_matches = (home_counts.add(away_counts, fill_value=0)).sort_values(ascending=False)

print('Top 15 most active international teams:')
print(total_matches.head(15).astype(int))

## 7. Load 2026 Data

In [ ]:
matches_2026 = pd.read_csv('../data/raw/wc2026/matches_detailed.csv')
players_2026 = pd.read_csv('../data/raw/wc2026/squads_and_players.csv')
elo = pd.read_csv('../data/raw/wc2026/elo_ratings_wc2026.csv')

print(f'2026 matches so far: {len(matches_2026)}')
print(f'Players registered: {len(players_2026)}')
print(f'Elo entries: {len(elo)}')

matches_2026.head()

## Next Steps
- [ ] Feature engineering notebook (form, head-to-head, Elo differential)
- [ ] Model training notebook (Win/Draw/Loss classifier)
- [ ] Monte Carlo simulation
- [ ] Player form scoring